In [1]:
import pandas    as pd
import panel     as pn
import numpy     as np
import hvplot.pandas
import os.path   as osp
import holoviews as hv
import nibabel   as nib
from nilearn.plotting import plot_stat_map
from nilearn.image    import index_img
import matplotlib.pyplot as plt

In [2]:
PRJ_DIR = '/data/SFIM_CSF_Volume/Project01'
PRCSDATA_DIR = osp.join(PRJ_DIR, 'PrcsData')

SBJ = 'SBJ06'
SES = 'SES01'
RUN = 'run3'
THIS_DATA_MELODIC_DIR = osp.join(PRCSDATA_DIR, SBJ, SES, RUN,'melodic_tica')
avail_signal_names = ['TI1,TE1','TI1,TE2','TI1,TE3','TI2,TE1','TI2,TE2','TI2,TE3','TI3,TE1','TI3,TE2','TI3,TE3']

# Load Melodica Tica Component Timeseries

In [3]:
ts_path = pd.read_csv(osp.join(THIS_DATA_MELODIC_DIR, 'melodic_mix'),header=None, sep='\\s+')
ts_df = pd.DataFrame(ts_path)
ts_df.columns = [f'IC{idx+1}' for idx in range(ts_df.shape[1])]
ts_df.columns.name='Component'
ts_df.index.name='TR'
print('++ Number of ICA components:', ts_df.shape[1])


++ Number of ICA components: 123


# Load Melodic Tica Spatial Modes

In [4]:
Smodes_path = pd.read_csv(osp.join(THIS_DATA_MELODIC_DIR, 'melodic_Smodes'),header=None, sep='\\s+')
Smodes_df = pd.DataFrame(Smodes_path)
Smodes_df.columns = [f'IC{idx+1}' for idx in range(ts_df.shape[1])]
Smodes_df.index = avail_signal_names
Smodes_df.index.name = 'Dataset'
Smodes_df.columns.name = 'Component'
print('++ Number of Spatial modes: %d (per component)' % Smodes_df.shape[0])

++ Number of Spatial modes: 9 (per component)


# Load Melodic Tica Temporal Modes

In [5]:
Tmodes_path       = pd.read_csv(osp.join(THIS_DATA_MELODIC_DIR, 'melodic_Tmodes'),header=None, sep='\\s+')
Tmodes_df         = pd.DataFrame(Tmodes_path)
Tmodes_df.shape

(233, 1230)

# Dashboard

In [6]:
avail_signal_names         = ['TI1,TE1','TI1,TE2','TI1,TE3','TI2,TE1','TI2,TE2','TI2,TE3','TI3,TE1','TI3,TE2','TI3,TE3']
avail_signal_labels_dict   = {'TI1,TE1':'TI1,TE1 (CSF Vol)','TI1,TE2':'TI1,TE2','TI1,TE3':'TI1,TE3',
                            'TI2,TE1':'TI2,TE1 CBV','TI2,TE2':'TI2,TE2','TI2,TE3':'TI2,TE3',
                            'TI3,TE1':'TI3,TE1','TI3,TE2':'TI3,TE2 BOLD','TI3,TE3':'TI3,TE3'}
ic_select                  = pn.widgets.Select(options=ts_df.columns.tolist(), name='ICA Component')
signal_01_select_id        = pn.widgets.Select(options=avail_signal_names, value=avail_signal_names[0],name='IC Segment A')
signal_02_select_id        = pn.widgets.Select(options=avail_signal_names+['None'], value=avail_signal_names[3],name='IC Segment B')
signal_03_select_id        = pn.widgets.Select(options=avail_signal_names+['None'], value=avail_signal_names[7],name='IC Segment C')
mult_segs_by_mode_checkbox = pn.widgets.Checkbox(name='Multiple Segments by Spatial Mode Weight', value=False)
show_spatial_map_checkbox  = pn.widgets.Checkbox(name='Show Spatial Map', value=False)
sidebar                    = pn.Column(ic_select,pn.layout.Divider(),signal_01_select_id,signal_02_select_id,signal_03_select_id,mult_segs_by_mode_checkbox,pn.layout.Divider(),show_spatial_map_checkbox)


In [7]:
@pn.depends(ic_select)
def plot_full_ts(ic):
    ts            = ts_df[ic]
    num_acqs      = ts.shape[0]
    num_scans     = 9
    x_vert_lines  = np.linspace(0, num_acqs, num_scans+1, dtype=int)
    x_tick_locs   = list(x_vert_lines[:-1] + np.floor(num_acqs/num_scans/2))
    x_tick_labels = ['TI1,TE1','TI1,TE2','TI1,TE3','TI2,TE1','TI2,TE2','TI2,TE3','TI3,TE1','TI3,TE2','TI3,TE3']
    x_tick_info   = [(int(x),l) for x,l in zip(x_tick_locs,x_tick_labels)]
    

    plot = ts.hvplot.line(
        title=f'Full Time Series for {ic}',
        xlabel='TR',
        ylabel='Signal',
        height=600,
        width=3000,
        grid=False,
        xticks=x_tick_info,
    ).opts(fontsize={'xticks':12,'yticks':16,'title':16,'xlabel':14,'ylabel':14})
    for x in x_vert_lines:
        plot = plot * hv.VLine(x).opts(color='black', line_dash='dashed',line_width=1)
    return plot

In [8]:
@pn.depends(ic_select, signal_01_select_id, signal_02_select_id, signal_03_select_id, mult_segs_by_mode_checkbox)
def plot_segmented_ts(ic, sig1, sig2, sig3, mult_segs_by_mode):
    ts            = ts_df[ic]
    ts            = ts_df[ic]
    num_acqs      = ts.shape[0]
    num_scans     = 9
    x_vert_lines  = np.linspace(0, num_acqs, num_scans+1, dtype=int)    
    plot = hv.Layout()
    seg1_idx_start = x_vert_lines[avail_signal_names.index(sig1)]
    seg1_idx_end   = x_vert_lines[avail_signal_names.index(sig1)+1]
    seg1_ts       = ts[seg1_idx_start:seg1_idx_end].reset_index(drop=True)
    if mult_segs_by_mode:
        seg1_ts = Smodes_df.loc[sig1,ic] * seg1_ts
    seg1_plot     = seg1_ts.hvplot.line(
    title=f'Segmented Time Series for {ic}',
        label=avail_signal_labels_dict[sig1],
        xlabel='TR',
        ylabel='Signal',
        height=400,
        width=1000,
        grid=True,
        shared_axes=False).opts(fontsize={'xticks':12,'yticks':16,'title':16,'xlabel':14,'ylabel':14})
    plot += seg1_plot
    if sig2 != 'None':
        seg2_idx_start = x_vert_lines[avail_signal_names.index(sig2)]
        seg2_idx_end   = x_vert_lines[avail_signal_names.index(sig2)+1]
        seg2_ts       = ts[seg2_idx_start:seg2_idx_end].reset_index(drop=True)
        if mult_segs_by_mode:
            seg2_ts = Smodes_df.loc[sig2,ic] * seg2_ts
        seg2_plot     = seg2_ts.hvplot.line(
            label=avail_signal_labels_dict[sig2],
            xlabel='TR',
            ylabel='Signal',
            height=400,
            width=1000,
            grid=False).opts(fontsize={'xticks':12,'yticks':16,'title':16,'xlabel':14,'ylabel':14})
        plot *= seg2_plot
    if sig3 != 'None':
        seg3_idx_start = x_vert_lines[avail_signal_names.index(sig3)]
        seg3_idx_end   = x_vert_lines[avail_signal_names.index(sig3)+1]
        seg3_ts       = ts[seg3_idx_start:seg3_idx_end].reset_index(drop=True)
        if mult_segs_by_mode:
            seg3_ts = Smodes_df.loc[sig3,ic] * seg3_ts
        seg3_plot     = seg3_ts.hvplot.line(
            label=avail_signal_labels_dict[sig3],
            xlabel='TR',
            ylabel='Signal',
            height=400,
            width=1000,
            grid=False).opts(fontsize={'xticks':12,'yticks':16,'title':16,'xlabel':14,'ylabel':14})
        plot *= seg3_plot
    return plot.opts(show_legends=True, legend_position='right')

In [9]:
@pn.depends(ic_select)
def plot_spatial_modes(ic):
    plot = (Smodes_df.loc[['TI1,TE1','TI1,TE2','TI1,TE3'],ic].hvplot.scatter(c='k', title=f'Spatial Modes for {ic}', width=400, height=400) * \
            Smodes_df.loc[['TI2,TE1','TI2,TE2','TI2,TE3'],ic].hvplot.scatter(c='r') * \
            Smodes_df.loc[['TI3,TE1','TI3,TE2','TI3,TE3'],ic].hvplot.scatter(c='b') *hv.HLine(0).opts(line_dash='dashed',line_width=1, line_color='gray')).opts(show_legend=False)
    return plot
    

In [10]:
@pn.depends(ic_select, show_spatial_map_checkbox)
def plot_ic_map(ic, show_map):
    ic_index = int(ic.replace('IC',''))-1
    if not show_map:
        return pn.pane.Markdown("### Spatial Map not selected to be shown.", width=600)
    else:
        fig,ax = plt.subplots(1,1, figsize=(12,4))
        ax.set_axis_off()
        ic_maps_path = osp.join(THIS_DATA_MELODIC_DIR, 'melodic_IC.nii.gz')
        this_ic_map  = index_img(ic_maps_path, ic_index)
        anat_path = osp.join(PRCSDATA_DIR, SBJ, SES, RUN,f'{RUN}_anat.unifize.nii')
        # Plot the statistical map
        this_ic_img = plot_stat_map(
                        this_ic_map,
                        bg_img=anat_path,
                        title="IC Map for %s" % ic,
                        threshold=0,  # Threshold for significance
                        alpha=0.5,    # Opacity of the overlay
                        colorbar=True,
                        figure=fig,
                    black_bg=False,
                    display_mode='mosaic')
        return pn.pane.Matplotlib(fig, tight=True)

In [11]:
main_content = pn.Column(plot_full_ts, pn.layout.Divider(), pn.Row(plot_segmented_ts,plot_spatial_modes),pn.layout.Divider(), plot_ic_map)

In [13]:
dashboard = pn.template.FastListTemplate(title=f"Melodic ICs Time Series: [{SBJ}, {SES}, {RUN}]", sidebar=sidebar, main=main_content).show()

Launching server at http://localhost:34059


# RUN UP TO HERE ONLY

In [16]:
dashboard.stop()

In [5]:
long = (
    melodic_ts.stack()               # stacks columns into rows
      .rename("value")       # name the stacked values
      .reset_index()         # turn index levels into columns
      .rename(columns={"level_1": "Component"})
)

In [6]:
# --- Interactive plot: dropdown to choose the component ---
plot = long.hvplot.line(
    x="TR",
    y="value",
    groupby="Component",     # <- creates the interactive widget
    width=1500,
    height=300,
    title="ICA component (choose from dropdown)",
    grid=True,
)

In [7]:
plot * \
    hv.VLine(233).opts(color='black', line_dash='dashed',line_width=1) * \
    hv.VLine(466).opts(color='black', line_dash='dashed',line_width=1) * \
    hv.VLine(699).opts(color='black',line_width=1) * \
    hv.VLine(932).opts(color='black', line_dash='dashed',line_width=1) * \
    hv.VLine(1165).opts(color='black', line_dash='dashed',line_width=1) * \
    hv.VLine(1398).opts(color='black', line_width=1) * \
    hv.VLine(1631).opts(color='black', line_dash='dashed',line_width=1) * \
    hv.VLine(1864).opts(color='black', line_dash='dashed',line_width=1)

BokehModel(combine_events=True, render_bundle={'docs_json': {'258f16f3-08e8-4002-985f-3cac807ceb5a': {'version…

In [116]:
plot = (melodic_ts['IC17'][0:233].reset_index(drop=True).hvplot(label='CSF Vol') * \
melodic_ts['IC17'][699:932].reset_index(drop=True).hvplot(label='CBV') * \
melodic_ts['IC17'][1864:-1].reset_index(drop=True).hvplot(label='BOLD')
)
plot.opts(show_legend=True, legend_position='right')


:Overlay
   .Curve.CSF_Vol :Curve   [index]   (IC17)
   .Curve.CBV     :Curve   [index]   (IC17)
   .Curve.BOLD    :Curve   [index]   (IC17)

In [117]:
type(plot)

holoviews.core.overlay.Overlay

In [9]:
import panel as pn

In [121]:
dashboard = pn.template.FastListTemplate(title="Melodic ICs Time Series", sidebar=sidebar, main=main_content).show()

Launching server at http://localhost:40979


ERROR:tornado.application:Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOMainLoop object at 0x155551db9fa0>>, <Task finished name='Task-48840' coro=<ServerSession.with_document_locked() done, defined at /data/SFIMJGC_HCP7T/Apps/envs/generic_2025a/lib/python3.12/site-packages/bokeh/server/session.py:77> exception=RuntimeError("Models must be owned by only a single document, Range1d(id='70cc1280-22ac-49e5-aa02-269ef78d7a07', ...) is already in a doc")>)
Traceback (most recent call last):
  File "/data/SFIMJGC_HCP7T/Apps/envs/generic_2025a/lib/python3.12/site-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/data/SFIMJGC_HCP7T/Apps/envs/generic_2025a/lib/python3.12/site-packages/tornado/ioloop.py", line 782, in _discard_future_result
    future.result()
  File "/data/SFIMJGC_HCP7T/Apps/envs/generic_2025a/lib/python3.12/site-packages/bokeh/server/session.py", li

In [106]:
dashboard.stop()

In [ ]:
def plot_